In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pyspark.sql.types as T
import os

In [2]:
app_name='PruebaBBVA'

In [3]:
path = "/Users/sqpr14/Desarrollo/Pyspark"

In [4]:
os.listdir(path)

['prueba.ipynb',
 '.DS_Store',
 'PAGO_2014 (1).txt',
 'guia.ipynb',
 '~$amen DS (1).docx',
 '1.ipynb',
 '.ipynb_checkpoints',
 'Examen DS (1).docx',
 'pyspark_0_tohero.pdf',
 'TIPO_CAMBIO (1).txt',
 'material']

In [5]:
spark = SparkSession.builder\
        .master("local[*]")\
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
        .appName(app_name)\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/02 20:19:52 WARN Utils: Your hostname, MacBook-Pro-de-Alberto.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.140 instead (on interface en0)
26/03/02 20:19:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 20:19:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
df = spark.read.csv(f'{path}/PAGO_2014 (1).txt', sep=';', header=True)

In [7]:
df.show()

+--------------+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+
|           RFC|EJERCICIO|PERIODO|FECHA_PRESENTACION|HORA_DE_PRESENTACION|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|
+--------------+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+
| BALL8606057R0|     2014|      1|        01/04/2014|         17:14:00.00|        13361703|         $0.00|     $6,809.00|            21|                 IVA|
|BALL860*6057R0|     2014|      2|        01/04/2014|         17:17:00.00|        13361717|         $0.00|     $2,328.00|            21|                 IVA|
| BALL8606057R0|     2014|      3|        25/04/2014|         13:09:00.00|        13499462|         $0.00|     $6,321.00|            21|                 IVA|
| BALL8606057R0|     2014|      4|        21/05/2014

In [8]:
df.printSchema()

root
 |-- RFC: string (nullable = true)
 |-- EJERCICIO: string (nullable = true)
 |-- PERIODO: string (nullable = true)
 |-- FECHA_PRESENTACION: string (nullable = true)
 |-- HORA_DE_PRESENTACION: string (nullable = true)
 |-- NUMERO_OPERACION: string (nullable = true)
 |-- IMPORTE_AFAVOR: string (nullable = true)
 |-- IMPORTE_ACARGO: string (nullable = true)
 |-- CLAVE_IMPUESTO: string (nullable = true)
 |-- DESCRIPCION_IMPUESTO: string (nullable = true)



In [9]:
df.count()

76

# Ejercicio 1

In [10]:
import re

df = df.withColumn("RFC_clean", F.substring(F.regexp_replace(F.col("RFC"), "[^a-zA-Z0-9]", ""), 1, 13))
df = df.drop("RFC")

In [11]:
df.show()

+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+-------------+
|EJERCICIO|PERIODO|FECHA_PRESENTACION|HORA_DE_PRESENTACION|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|
+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+-------------+
|     2014|      1|        01/04/2014|         17:14:00.00|        13361703|         $0.00|     $6,809.00|            21|                 IVA|BALL8606057R0|
|     2014|      2|        01/04/2014|         17:17:00.00|        13361717|         $0.00|     $2,328.00|            21|                 IVA|BALL8606057R0|
|     2014|      3|        25/04/2014|         13:09:00.00|        13499462|         $0.00|     $6,321.00|            21|                 IVA|BALL8606057R0|
|     2014|      4|        21/05/2014|         13:05:00.00

In [12]:
df.filter(F.length(F.col("RFC_clean")) > 13).show()

+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+---------+
|EJERCICIO|PERIODO|FECHA_PRESENTACION|HORA_DE_PRESENTACION|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|RFC_clean|
+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+---------+
+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+---------+



#### Ningun registro del RFC es mayor a 13 y con la REGEX nos aseguramos que no dejó pasar ningún caracter que no sea alfanumérico de la manera menos costosa computacionalmente hablando.

# Ejercicio 2

In [13]:
df = df.withColumn("tmsp_aux", F.concat_ws(" ", F.col("FECHA_PRESENTACION"), F.col("HORA_DE_PRESENTACION")))
df = df.withColumn("TIMESTAMP", F.to_timestamp(F.col("tmsp_aux"), "dd/MM/yyyy HH:mm:ss.SS"))
df.show()

+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+-------------+--------------------+-------------------+
|EJERCICIO|PERIODO|FECHA_PRESENTACION|HORA_DE_PRESENTACION|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|            tmsp_aux|          TIMESTAMP|
+---------+-------+------------------+--------------------+----------------+--------------+--------------+--------------+--------------------+-------------+--------------------+-------------------+
|     2014|      1|        01/04/2014|         17:14:00.00|        13361703|         $0.00|     $6,809.00|            21|                 IVA|BALL8606057R0|01/04/2014 17:14:...|2014-04-01 17:14:00|
|     2014|      2|        01/04/2014|         17:17:00.00|        13361717|         $0.00|     $2,328.00|            21|                 IVA|BALL8606057R0|01/04/2014 17:17:...|2014-04-01 17:17:00|
|     2014

In [14]:
l = ['FECHA_PRESENTACION', 'HORA_DE_PRESENTACION', 'tmsp_aux']
df = df.drop(*l)
df.show()

+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+
|EJERCICIO|PERIODO|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|          TIMESTAMP|
+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+
|     2014|      1|        13361703|         $0.00|     $6,809.00|            21|                 IVA|BALL8606057R0|2014-04-01 17:14:00|
|     2014|      2|        13361717|         $0.00|     $2,328.00|            21|                 IVA|BALL8606057R0|2014-04-01 17:17:00|
|     2014|      3|        13499462|         $0.00|     $6,321.00|            21|                 IVA|BALL8606057R0|2014-04-25 13:09:00|
|     2014|      4|        13656923|         $0.00|       $719.00|             7|ISR personas fisi...|BALL8606057R0|2014-05-21 13:05:00|
|     2014|      4|        13656923|     

In [18]:
window_aa = Window.partitionBy("RFC_clean", "EJERCICIO", "PERIODO", "CLAVE_IMPUESTO").orderBy(F.col("TIMESTAMP").asc())
df_ordenado = df.withColumn("orden_declaracion", F.row_number().over(window_asc))
df_complementarias = df_ordenado.filter(F.col("orden_declaracion") > 1)
df_complementarias.show()

+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+-----------------+
|EJERCICIO|PERIODO|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|          TIMESTAMP|orden_declaracion|
+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+-----------------+
|     2014|      1|        13361704|$15,000,000.00|         $0.00|             7|ISR personas fisi...|BALL8606057R0|2014-05-21 13:05:00|                2|
|     2014|      3|        13460732|         $0.00|    $20,253.00|            24|     IVA retenciones| BPI9205273M1|2014-04-23 17:32:00|                2|
+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+-----------------+



In [22]:
window_b = Window.partitionBy("RFC_clean", "EJERCICIO", "PERIODO", "CLAVE_IMPUESTO").orderBy(F.col("TIMESTAMP").desc())
df_ultima_declaracion = df.withColumn("fila", F.row_number().over(window_b)).filter(F.col("fila") == 1)
df_final = df_ultima_declaracion.drop("TIMESTAMP", "fila")
df_final.show(truncate=False)

+---------+-------+----------------+--------------+--------------+--------------+---------------------------------------------------------+-------------+
|EJERCICIO|PERIODO|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO                                     |RFC_clean    |
+---------+-------+----------------+--------------+--------------+--------------+---------------------------------------------------------+-------------+
|2014     |1      |13361703        |$0.00         |$6,809.00     |21            |IVA                                                      |BALL8606057R0|
|2014     |1      |13361704        |$15,000,000.00|$0.00         |7             |ISR personas fisicas. Actividad empresarial y profesional|BALL8606057R0|
|2014     |10     |14849643        |$0.00         |$6,631.00     |21            |IVA                                                      |BALL8606057R0|
|2014     |10     |14849643        |$0.00         |$971.00       |7         

# Ejercicio 3

In [30]:
df.filter((F.col("RFC_clean") == "BALL8606057R0") & (F.col("CLAVE_IMPUESTO") == 7)).show()

+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+
|EJERCICIO|PERIODO|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|          TIMESTAMP|
+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+-------------------+
|     2014|      4|        13656923|         $0.00|       $719.00|             7|ISR personas fisi...|BALL8606057R0|2014-05-21 13:05:00|
|     2014|      5|        13897722|         $0.00|     $3,096.00|             7|ISR personas fisi...|BALL8606057R0|2014-07-02 16:33:00|
|     2014|      6|        14073090|         $0.00|     $2,332.00|             7|ISR personas fisi...|BALL8606057R0|2014-08-01 18:06:00|
|     2014|      7|        14343603|         $0.00|     $1,488.00|             7|ISR personas fisi...|BALL8606057R0|2014-09-19 13:11:00|
|     2014|      8|        14481664|     

In [38]:
df_aux = df_final.withColumn("cargo_num", F.regexp_replace(F.col("IMPORTE_ACARGO"), r"[\$,]", "").cast("float"))\
    .withColumn("favor_num", F.regexp_replace(F.col("IMPORTE_AFAVOR"), r"[\$,]", "").cast("float"))

In [39]:
df_f = df_aux.filter((F.col("RFC_clean") == "BALL8606057R0") & (F.col("CLAVE_IMPUESTO") == 7))

In [40]:
df_f.show()

+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+---------+---------+
|EJERCICIO|PERIODO|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|cargo_num|favor_num|
+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+---------+---------+
|     2014|      1|        13361704|$15,000,000.00|         $0.00|             7|ISR personas fisi...|BALL8606057R0|      0.0|    1.5E7|
|     2014|     10|        14849643|         $0.00|       $971.00|             7|ISR personas fisi...|BALL8606057R0|    971.0|      0.0|
|     2014|      4|        13656923|         $0.00|       $719.00|             7|ISR personas fisi...|BALL8606057R0|    719.0|      0.0|
|     2014|      5|        13897722|         $0.00|     $3,096.00|             7|ISR personas fisi...|BALL8606057R0|   3096.0|      0.0|
|     2014|      6|        14073090|     

In [46]:
df_s = df_f.withColumn("saldo", F.col("favor_num") - F.col("cargo_num"))

In [47]:
df_s.show()

+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+---------+---------+-------+
|EJERCICIO|PERIODO|NUMERO_OPERACION|IMPORTE_AFAVOR|IMPORTE_ACARGO|CLAVE_IMPUESTO|DESCRIPCION_IMPUESTO|    RFC_clean|cargo_num|favor_num|  saldo|
+---------+-------+----------------+--------------+--------------+--------------+--------------------+-------------+---------+---------+-------+
|     2014|      1|        13361704|$15,000,000.00|         $0.00|             7|ISR personas fisi...|BALL8606057R0|      0.0|    1.5E7|  1.5E7|
|     2014|     10|        14849643|         $0.00|       $971.00|             7|ISR personas fisi...|BALL8606057R0|    971.0|      0.0| -971.0|
|     2014|      4|        13656923|         $0.00|       $719.00|             7|ISR personas fisi...|BALL8606057R0|    719.0|      0.0| -719.0|
|     2014|      5|        13897722|         $0.00|     $3,096.00|             7|ISR personas fisi...|BALL8606057R0|   3096.0|    

In [48]:
res = df_s.agg(F.sum("saldo").alias("saldo_total_ISR"))

In [50]:
res.show()

+---------------+
|saldo_total_ISR|
+---------------+
|    1.4989718E7|
+---------------+



In [54]:
res.select(F.format_number("saldo_total_ISR", 2).alias("saldo_total_ISR")).show()

+---------------+
|saldo_total_ISR|
+---------------+
|  14,989,718.00|
+---------------+



# Ejercicio 5

In [56]:
df_5 = df.groupBy("RFC_clean", "CLAVE_IMPUESTO").agg((F.countDistinct("PERIODO") / 12).alias("nivel_cumplimiento"))

df_5.show()

+-------------+--------------+------------------+
|    RFC_clean|CLAVE_IMPUESTO|nivel_cumplimiento|
+-------------+--------------+------------------+
|BALL8606057R0|            21|0.8333333333333334|
|BALL8606057R0|             7|0.6666666666666666|
| BPI9205273M1|            14|0.9166666666666666|
|BASS4405066HA|             7|              0.25|
| BPI9205273M1|            15|0.9166666666666666|
| BPI9205273M1|            24|0.9166666666666666|
|BASS4405066HA|            21|0.5833333333333334|
| BPI9205273M1|            21|              0.75|
|BASS4405066HA|            12|0.3333333333333333|
+-------------+--------------+------------------+



# Ejercicio 6

In [58]:
def generar_tabla_fechas(FECHA, DIAS, ITERACIONES, spark_session):
    """
    Función que genera una tabla de fechas sumando 'n' días iterativamente.
    """
    dias_totales = DIAS * (ITERACIONES - 1)
    query = f"""
        SELECT explode(
            sequence(
                to_date('{FECHA}', 'yyyy-MM-dd'), 
                date_add(to_date('{FECHA}', 'yyyy-MM-dd'), {dias_totales}), 
                INTERVAL {DIAS} DAY
            )
        ) AS FECHA_GENERADA
    """
    df_fechas = spark_session.sql(query)
    return df_fechas

In [59]:
FECHA = '2014-01-01'
DIAS = 3             
ITERACIONES = 7

In [60]:
df_resultado = generar_tabla_fechas(FECHA, DIAS, ITERACIONES, spark)

In [61]:
df_resultado.show()

+--------------+
|FECHA_GENERADA|
+--------------+
|    2014-01-01|
|    2014-01-04|
|    2014-01-07|
|    2014-01-10|
|    2014-01-13|
|    2014-01-16|
|    2014-01-19|
+--------------+

